In [13]:
# Core libraries
import pandas as pd
import numpy as np

# ML libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

# Serialization
import joblib

In [14]:
# Load dataset
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

# Data validation checks
print("Shape:", df.shape)
print("Missing values:\n", df.isnull().sum().sum())
print("Data types:\n", df.dtypes.head())

Shape: (569, 31)
Missing values:
 0
Data types:
 mean radius        float64
mean texture       float64
mean perimeter     float64
mean area          float64
mean smoothness    float64
dtype: object


In [15]:
# Basic statistics
print(df.describe())

# Class distribution
class_distribution = df["target"].value_counts()
print(class_distribution)

       mean radius  mean texture  mean perimeter    mean area  \
count   569.000000    569.000000      569.000000   569.000000   
mean     14.127292     19.289649       91.969033   654.889104   
std       3.524049      4.301036       24.298981   351.914129   
min       6.981000      9.710000       43.790000   143.500000   
25%      11.700000     16.170000       75.170000   420.300000   
50%      13.370000     18.840000       86.240000   551.100000   
75%      15.780000     21.800000      104.100000   782.700000   
max      28.110000     39.280000      188.500000  2501.000000   

       mean smoothness  mean compactness  mean concavity  mean concave points  \
count       569.000000        569.000000      569.000000           569.000000   
mean          0.096360          0.104341        0.088799             0.048919   
std           0.014064          0.052813        0.079720             0.038803   
min           0.052630          0.019380        0.000000             0.000000   
25%      

In [16]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_prob = lr.predict_proba(X_test_scaled)[:,1]

In [18]:
dt = DecisionTreeClassifier(max_depth=5)

dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_prob = dt.predict_proba(X_test)[:,1]

In [19]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

In [20]:
gb = GradientBoostingClassifier()

gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_prob = gb.predict_proba(X_test)[:,1]

In [21]:
cv_scores = cross_val_score(rf, X, y, cv=5)

print("Cross-validation scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())

Cross-validation scores: [0.92105263 0.93859649 0.98245614 0.96491228 0.97345133]
Mean CV Score: 0.9560937742586555


In [22]:
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=3)
grid.fit(X_train, y_train)

best_rf = grid.best_estimator_

In [23]:
def evaluate_model(name, y_test, y_pred, y_prob):
    return [
        name,
        accuracy_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ]

results = []

results.append(evaluate_model("Logistic Regression", y_test, lr_pred, lr_prob))
results.append(evaluate_model("Decision Tree", y_test, dt_pred, dt_prob))
results.append(evaluate_model("Random Forest", y_test, rf_pred, rf_prob))
results.append(evaluate_model("Gradient Boosting", y_test, gb_pred, gb_prob))

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1", "AUC"])
results_df

,Model,Accuracy,F1,AUC
0,Logistic Regression,0.982456,0.986111,0.995370
1,Decision Tree,0.929825,0.943662,0.923446
2,Random Forest,0.956140,0.965517,0.993717
3,Gradient Boosting,0.956140,0.965986,0.990741


In [24]:
# Slight perturbation test
X_train_noisy = X_train + np.random.normal(0, 0.01, X_train.shape)

rf_noisy = RandomForestClassifier(n_estimators=100)
rf_noisy.fit(X_train_noisy, y_train)

rf_noisy_pred = rf_noisy.predict(X_test)

stability_score = accuracy_score(rf_pred, rf_noisy_pred)
print("Stability score:", stability_score)

Stability score: 0.9912280701754386


In [25]:
joblib.dump(rf, "random_forest_model.pkl")
joblib.dump(lr, "logistic_model.pkl")
joblib.dump(dt, "decision_tree_model.pkl")
joblib.dump(gb, "gradient_boosting_model.pkl")

['gradient_boosting_model.pkl']

In [26]:
results_df.to_csv("dashboard1_model_performance.csv", index=False)

In [27]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.to_csv("dashboard2_feature_importance.csv", index=False)

In [28]:
cm = confusion_matrix(y_test, rf_pred)

cm_df = pd.DataFrame(cm,
                     columns=["Predicted 0", "Predicted 1"],
                     index=["Actual 0", "Actual 1"])

cm_df.to_csv("dashboard3_confusion_matrix.csv")

In [29]:
class_dist = df["target"].value_counts().reset_index()
class_dist.columns = ["Class", "Count"]

class_dist.to_csv("dashboard4_class_distribution.csv", index=False)